# Exploratory Data Analysis for Modelling

Notebook for **Chapter 3** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch03.html

Tip: change a number, predict the output, then run the cell again.

In [ ]:
from ds_common import *
from sklearn.metrics import roc_auc_score, mutual_info_score

c, o = make_data()
rng = np.random.default_rng(11)

def features(T):
    d = make_label_table(c, o, pred_date=T)
    past = o[o.order_ts < T]
    w = past[past.order_ts >= T - pd.Timedelta(days=90)]
    g = w.groupby("customer_id")
    f = pd.DataFrame({"orders_90d": g.size(), "spend_90d": g.amount.sum(),
                      "avg_order": g.amount.mean(),
                      "recency_days": (T - past.groupby("customer_id").order_ts.max()).dt.days})
    d = d.merge(f, left_on="customer_id", right_index=True, how="left")
    d["tenure_days"] = (T - d.signup_date).dt.days
    return d

d = features(pd.Timestamp("2025-01-01"))
d["age_raw"] = d.age.copy()
d.loc[rng.random(len(d)) < 0.07, "age_raw"] = -1                  # planted: default value
miss_p = np.where(d.churn == 1, 0.18, 0.05)
d.loc[rng.random(len(d)) < miss_p, "age_raw"] = np.nan            # planted: informative missingness
d["cancel_ticket"] = ((d.churn == 1) & (rng.random(len(d)) < 0.55)).astype(int) \
                     | ((d.churn == 0) & (rng.random(len(d)) < 0.01)).astype(int)  # planted: created AFTER T

## 3.1 Profiling and Missing Values

In [ ]:
print("shape", d.shape, " memory MB", round(d.memory_usage(deep=True).sum() / 1e6, 2))
print(d[["plan", "channel", "city"]].nunique().to_dict(), " duplicate ids:", d.customer_id.duplicated().sum())
print(d[["age_raw", "orders_90d", "spend_90d", "avg_order", "recency_days", "tenure_days"]]
      .describe().round(1).T[["count", "mean", "50%", "max"]])

shape (2562, 15)  memory MB 0.66
{'plan': 2, 'channel': 3, 'city': 6}  duplicate ids: 0
               count    mean     50%      max
age_raw         2380.0    30.1    32.0     61.0
orders_90d      2562.0     9.2      7.0     67.0
spend_90d       2562.0  5260.5   3512.9  41356.7
avg_order       2562.0   548.4    528.7   2840.2
recency_days    2562.0    19.4      9.0     90.0
tenure_days     2562.0   258.4    210.0    730.0


In [ ]:
print(d[["age_raw"]].isna().groupby(d.churn).mean().round(3)
        .rename(columns={"age_raw": "age_missing_rate"}))
print("share with age == -1:", round((d.age_raw == -1).mean(), 3))

       age_missing_rate
churn                  
0                 0.044
1                 0.149


## 3.2 One Column at a Time, Then Against the Target

In [ ]:
d["rec_bin"] = pd.cut(d.recency_days, [0, 7, 14, 30, 60, 91])
for col in ["plan", "channel", "city"]:
    print(d.groupby(col).churn.agg(["mean", "size"]).round(3).T.to_string())
print(d.groupby("rec_bin").churn.agg(["mean", "size"]).round(3).T.to_string())

plan      free      pro
mean     0.313     0.17
size  1509.000  1053.00
channel      ads   organic  referral
mean       0.309     0.258     0.157
size     726.000  1372.000   464.000
city  Bengaluru  Chennai    Delhi  Mumbai    Other     Pune
mean       0.244    0.252    0.243    0.27    0.238    0.295
size     504.000  246.000  497.000  525.00  526.000  264.000
rec_bin    (0, 7]  (7, 14]  (14, 30]  (30, 60]  (60, 91]
mean        0.047    0.127     0.292      0.63     0.899
size  1118.000  474.000   401.000    332.00   237.000


In [ ]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b).values; n = ct.sum()
    exp = ct.sum(1, keepdims=True) * ct.sum(0, keepdims=True) / n
    chi2 = ((ct - exp) ** 2 / exp).sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print("Cramer's V:", {k: round(float(cramers_v(d[k], d.churn)), 3) for k in ["plan", "channel", "city"]})
print("MI (nats) rec_bin:", round(mutual_info_score(d.rec_bin.astype(str), d.churn), 4))
print(d[["churn", "orders_90d", "spend_90d", "recency_days", "tenure_days"]]
      .corr(method="spearman").round(2)["churn"].to_dict())

Cramer's V: {'plan': 0.161, 'channel': 0.116, 'city': 0.042}
MI (nats) rec_bin: 0.2042
{'churn': 1.0, 'orders_90d': -0.41, 'spend_90d': -0.39, 'recency_days': 0.57, 'tenure_days': 0.01}


## 3.3 Segments, Time and Simpson's Check

In [ ]:
rows = []
for T in pd.to_datetime(["2024-04-01", "2024-07-01", "2024-10-01", "2025-01-01", "2025-02-28"]):
    t = features(T)
    rows.append((T.date(), len(t), round(t.churn.mean(), 3), round(t.recency_days.mean(), 1), round(t.orders_90d.mean(), 1)))
print(pd.DataFrame(rows, columns=["T", "rows", "churn_rate", "mean_recency", "mean_orders90"]).to_string(index=False))

         T  rows  churn_rate  mean_recency  mean_orders90
2024-04-01  2124       0.249          19.2            8.5
2024-07-01  2258       0.256          19.4            8.8
2024-10-01  2424       0.239          18.5            9.2
2025-01-01  2562       0.254          19.4            9.2
2025-02-28  2275       0.244          20.0           10.2


In [ ]:
print(d.groupby(["channel", "plan"]).churn.mean().unstack().round(3))

plan         free    pro
channel                 
ads         0.402  0.192
organic     0.319  0.168
referral    0.171  0.136


## 3.4 Leakage Hunting and Writing It Up

In [ ]:
rows = []
for col in ["orders_90d", "spend_90d", "avg_order", "recency_days", "tenure_days", "cancel_ticket"]:
    x = d[col].fillna(d[col].median())
    a = roc_auc_score(d.churn, x)
    rows.append((col, round(max(a, 1 - a), 3)))
print(pd.DataFrame(rows, columns=["feature", "single_feature_AUC"])
        .sort_values("single_feature_AUC", ascending=False).to_string(index=False))
print(d.groupby("cancel_ticket").churn.agg(["mean", "size"]).round(3))
print("ticket share among churners:", round(d[d.churn==1].cancel_ticket.mean(), 3))

      feature  single_feature_AUC
 recency_days               0.880
   orders_90d               0.770
    spend_90d               0.761
cancel_ticket               0.753
    avg_order               0.576
  tenure_days               0.505
                mean  size
cancel_ticket                 
0              0.143  2206
1              0.944   356
ticket share among churners: 0.516


## Chapter recap

## Practice set

Problem 3.P.1 (easy) 182 of 2,562 ages are missing. What percent? **Answer:** 182 / 2,562 = 7.1%.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch03.html